In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F 
from torch.autograd import Variable
import numpy as np
import cv2

In [2]:
def parse_cfg(cfgfile):
    """
    input: a configuration file, eg. yolov3.cfg
    
    Returns: blocks, which is a list of blocks. 
        -- each block corresponds to a block in cfg file 
        -- each block is represented as a dictionary
    """
    
    file = open(cfgfile, 'r')
    #lines = file.read().split('\n')                # store the lines in a list
    lines = file.read().splitlines()                # store the lines in a list
    lines = [x for x in lines if len(x) > 0]        # remove the empty lines 
    lines = [x for x in lines if x[0] != '#']       # remove the comment lines
    lines = [x.rstrip().lstrip() for x in lines]    # remove leading and trailing whitespaces
    #print(lines)
    # lines: a list of strings, each string is an effective line in cfg
    
    block = {}          # inital a dict
    blocks = []         # initial list
    
    for line in lines:
        if line[0] == "[":      # line is a string, line[0] is the first character in the line
            if len(block) != 0:          
                # If block is not empty, 
                # the "block" is for the previous block and ready to store to "blocks"
                blocks.append(block)     # add it the blocks list
                block = {}               # re-init the block
            block["type"] = line[1:-1].rstrip()  # store the block "type" for current block  
        else:
            key,value = line.split("=") 
            block[key.rstrip()] = value.lstrip() # store "value" to block[key]
    blocks.append(block)            # store the last "block" to the list     
    
    return blocks


In [3]:
blocks = parse_cfg("cfg/yolov3.cfg")
#print(blocks)

In [4]:
len(blocks)

108

In [5]:
class EmptyLayer(nn.Module):
    def __init__(self):
        super(EmptyLayer, self).__init__()
        

class DetectionLayer(nn.Module):
    def __init__(self, anchors):
        super(DetectionLayer, self).__init__()
        self.anchors = anchors

def create_modules(blocks):
    """
    input: a list -- blocks, each element is a dict -- block
    returns: net_info -- the first block about the net
             module_list: a list of nn.modules
    """
    net_info = blocks[0]           # net_info is a dict for the net information    
    module_list = nn.ModuleList()  # init a list containing nn.modules
    prev_filters = 3               # initial prev_filters
    output_filters = []            # store 
    
    for index, x in enumerate(blocks[1:]):
        module = nn.Sequential()    # initial module for current block
        
        #start with blocks[1]
        #check the type of block
        #create a new module for the block
        #append to module_list
        
        #If a convolutional layer
        if (x["type"] == "convolutional"):
            #Get the info about the layer
            activation = x["activation"]
            try:
                batch_normalize = int(x["batch_normalize"])
                bias = False
            except:
                batch_normalize = 0
                bias = True
        
            filters= int(x["filters"])
            padding = int(x["pad"])         # whether zero padding
            kernel_size = int(x["size"])
            stride = int(x["stride"])
        
            if padding:
                pad = (kernel_size - 1) // 2     # pad: the zero pad in nn.Conv2d()
            else:
                pad = 0
        
            #Add the convolutional layer
            conv = nn.Conv2d(prev_filters, filters, kernel_size, stride, pad, bias = bias)
            module.add_module("conv_{0}".format(index), conv)
        
            #Add the Batch Norm Layer
            if batch_normalize:
                bn = nn.BatchNorm2d(filters)
                module.add_module("batch_norm_{0}".format(index), bn)
        
            #Check the activation. 
            if activation == "leaky":
                activn = nn.LeakyReLU(0.1, inplace = True)
                module.add_module("leaky_{0}".format(index), activn)
        
        # If an upsampling layer
        # nearest mode, factor = stride
        elif (x["type"] == "upsample"):
            stride = int(x["stride"])
            upsample = nn.Upsample(scale_factor = stride, mode = "nearest")
            module.add_module("upsample_{}".format(index), upsample)
            # filters is the same as the previous layer, no need to update
                
        #If a route layer
        elif (x["type"] == "route"):
            
            route = EmptyLayer()
            module.add_module("route_{0}".format(index), route)
            
            x["layers"] = x["layers"].split(',')
            #Start  of a route
            start = int(x["layers"][0])   # the first integer e.g -1, or -4
            
            #end, if there is a second value for layers, e.g, 36, or 61.
            try:
                end = int(x["layers"][1])  # second value
            except:
                end = 0    # no second value
                
            if start > 0:
                start = start - index
            if end > 0:
                end = end - index
            
            if end == 0:   # only one value (start)
                filters = output_filters[index + start]
            else:  # two values (start, end)
                filters = output_filters[index + start] + output_filters[index + end]
                    
                
        #shortcut corresponds to skip connection
        elif x["type"] == "shortcut":
            shortcut = EmptyLayer()
            module.add_module("shortcut_{}".format(index), shortcut)
            # no need to update filters
            
        #Yolo is the detection layer
        elif x["type"] == "yolo":
            mask = x["mask"].split(",")
            mask = [int(x) for x in mask]
    
            anchors = x["anchors"].split(",")
            anchors = [int(a) for a in anchors]
            anchors = [(anchors[i], anchors[i+1]) for i in range(0, len(anchors),2)]
            anchors = [anchors[i] for i in mask]
    
            detection = DetectionLayer(anchors)
            module.add_module("Detection_{}".format(index), detection)
            # no need to update filters
                              
        module_list.append(module)          # add module to the module_list
        prev_filters = filters              
        # will be used for the input channels if the next layer is conv2d
        
        output_filters.append(filters)     
        # save the channels for layers to be used in layer "route"
        
    return (net_info, module_list)


In [6]:
net_info, module_list = create_modules(blocks)
len(module_list)

107

In [7]:
def predict_transform(prediction, inp_dim, anchors, num_classes):
    # this function is used in yolo DetectionLayer()
    # generates all predictions for one grid scale
    #   1) re-organize the tensor
    #   2) scale the anchors
    #   3) coordinate transform
    #   4) apply sigmoid for class probabilities
    
    """
    inputs: 
        -- prediction: output from the last conv2d
                shape [batch, 255, grid_size, grid_size] (e.g.[b, 255,13,13])
        -- inp_dim: input size, integer, e.g. 416
        -- anchors: a list of 3 anchors for this grid scale, [(*,*), (*,*), (*,*)]
        -- num_classes: the number of classes, integer, e.g. 80
    returns: 
        -- prediction: tensor shape [batch, grid_size x grid_size x 3, 85]
                       each box: prediction[i,j,:] =(bx,by,bw,bh,sigmoid(to), p1, ..., p80)
                       coordinates in unit of pixel
    """
    
    batch_size = prediction.size(0)
    stride =  inp_dim // prediction.size(2)
    grid_size = prediction.size(2)
    bbox_attrs = 5 + num_classes
    num_anchors = len(anchors)
    
    prediction = prediction.view(batch_size, bbox_attrs*num_anchors, grid_size*grid_size)
    #example: [b, 255,13,13]--> [b, 255, 169]
    
    prediction = prediction.transpose(1,2).contiguous()
    #example: [b,255,169] --> [b,169,255]
    
    prediction = prediction.view(batch_size, grid_size*grid_size*num_anchors, bbox_attrs)
    #example: [b,169,255] --> [b,169x3,85]=[b,507,85]
    
    anchors = [(a[0]/stride, a[1]/stride) for a in anchors]
    # scaled to unit of grid size

    #Sigmoid the  centre_X, centre_Y. and object confidencce
    prediction[:,:,0] = torch.sigmoid(prediction[:,:,0])
    prediction[:,:,1] = torch.sigmoid(prediction[:,:,1])
    prediction[:,:,4] = torch.sigmoid(prediction[:,:,4])
    
    #Add the center offsets
    grid = np.arange(grid_size)
    a,b = np.meshgrid(grid, grid)   # a: (13,13), b: (13,13)

    x_offset = torch.FloatTensor(a).view(-1,1)  #shape(169,1)
    y_offset = torch.FloatTensor(b).view(-1,1)  #shape(169,1)

    x_y_offset = torch.cat((x_offset, y_offset), 1).repeat(1,num_anchors).view(-1,2).unsqueeze(0)
    # x_y_offset, shape (1, 507, 2), grid coordinates repeated 3 time. 
    # [[0.,0.], [0., 0.], [0.,0.], 
    #  [1.,0.], [1.,0.], [1.,0.],
    #  [2.,0.], [2.,0.], [2.,0.],
    #  ....
    #  [12.,12.], [12.,12.], [12.,12.]]

    prediction[:,:,:2] += x_y_offset    
    # bounding box: bx, by

    #log space transform height and the width
    anchors = torch.FloatTensor(anchors)


    anchors = anchors.repeat(grid_size*grid_size, 1).unsqueeze(0)  
    # shape: (1, 507, 2) to match prediction (1, 507, 85)
    
    prediction[:,:,2:4] = torch.exp(prediction[:,:,2:4])*anchors
    # bounding box: bw, bh
    
    prediction[:,:,5: 5 + num_classes] = torch.sigmoid((prediction[:,:, 5 : 5 + num_classes]))

    prediction[:,:,:4] *= stride   
    # scaled back to unit of pixel, shape is [batch, 13x13x3, 85]
    # one bounding box prediction[1,1,:] is (bx,by,bw,bh,sigmoid(to), p1, p2, ..., p80)
    
    return prediction


In [8]:
class Yolonet(nn.Module):
    def __init__(self, cfgfile):
        super(Yolonet, self).__init__()
        self.blocks = parse_cfg(cfgfile)
        self.net_info, self.module_list = create_modules(self.blocks)
        
    def forward(self, x):
        modules = self.blocks[1:]
        outputs = {}   # We save the outputs for the route or shortcut layer, 
                       # key is layer index, value is the output of the layer
        
        first = 1
        for i, module in enumerate(modules):        
            module_type = (module["type"])
            
            if module_type == "convolutional" or module_type == "upsample":
                x = self.module_list[i](x)
    
            elif module_type == "route":
                layers = module["layers"]
                layers = [int(a) for a in layers]
    
                if (layers[0]) > 0:
                    layers[0] = layers[0] - i
    
                if len(layers) == 1:
                    x = outputs[i + (layers[0])]
    
                else:
                    if (layers[1]) > 0:
                        layers[1] = layers[1] - i
    
                    map1 = outputs[i + layers[0]]
                    map2 = outputs[i + layers[1]]
                    x = torch.cat((map1, map2), 1)
                
    
            elif  module_type == "shortcut":
                from_ = int(module["from"])
                x = outputs[i-1] + outputs[i+from_]
    
            elif module_type == 'yolo':        
                anchors = self.module_list[i][0].anchors
                #Get the input dimensions
                inp_dim = int (self.net_info["height"])
        
                #Get the number of classes
                num_classes = int (module["classes"])
        
                #Transform 
                x = x.data
                x = predict_transform(x, inp_dim, anchors, num_classes)
                if first:              #if the first yolo detectio head. 
                    detections = x
                    first = 0
        
                else:       # if the 2nd and 3rd head
                    detections = torch.cat((detections, x), 1)
        
            outputs[i] = x
        
        return detections


    def load_weights(self, weightfile):
        # Open weightfile
        fp = open(weightfile, "rb")
    
        #The first 5 values are header information 
        header = np.fromfile(fp, dtype = np.int32, count = 5)
        self.header = torch.from_numpy(header)
        self.seen = self.header[3]   
        
        weights = np.fromfile(fp, dtype = np.float32)
        
        ptr = 0
        for i in range(len(self.module_list)):
            module_type = self.blocks[i + 1]["type"]
    
            # If module_type is convolutional load weights
            # Otherwise do nothing.
            
            if module_type == "convolutional":
                model = self.module_list[i]
                try:
                    batch_normalize = int(self.blocks[i+1]["batch_normalize"])
                except:
                    batch_normalize = 0
            
                conv = model[0]
                
                 
                if (batch_normalize):   # if BN applied, load BN parameters
                    bn = model[1]
        
                    #Get the number of weights of Batch Norm Layer
                    num_bn_biases = bn.bias.numel()
        
                    #Load the weights
                    bn_biases = torch.from_numpy(weights[ptr:ptr + num_bn_biases])
                    ptr += num_bn_biases
        
                    bn_weights = torch.from_numpy(weights[ptr: ptr + num_bn_biases])
                    ptr  += num_bn_biases
        
                    bn_running_mean = torch.from_numpy(weights[ptr: ptr + num_bn_biases])
                    ptr  += num_bn_biases
        
                    bn_running_var = torch.from_numpy(weights[ptr: ptr + num_bn_biases])
                    ptr  += num_bn_biases
        
                    #Cast the loaded weights into dims of model weights. 
                    bn_biases = bn_biases.view_as(bn.bias.data)
                    bn_weights = bn_weights.view_as(bn.weight.data)
                    bn_running_mean = bn_running_mean.view_as(bn.running_mean)
                    bn_running_var = bn_running_var.view_as(bn.running_var)
        
                    #Copy the data to model
                    bn.bias.data.copy_(bn_biases)
                    bn.weight.data.copy_(bn_weights)
                    bn.running_mean.copy_(bn_running_mean)
                    bn.running_var.copy_(bn_running_var)
                
                else: # if no BN, load conv layer biases
                    #Number of biases
                    num_biases = conv.bias.numel()
                
                    #Load the weights
                    conv_biases = torch.from_numpy(weights[ptr: ptr + num_biases])
                    ptr = ptr + num_biases
                
                    #reshape the loaded weights according to the dims of the model weights
                    conv_biases = conv_biases.view_as(conv.bias.data)
                
                    #Finally copy the data
                    conv.bias.data.copy_(conv_biases)
                    
                #load the weights for the Convolutional layers
                num_weights = conv.weight.numel()
                conv_weights = torch.from_numpy(weights[ptr:ptr+num_weights])
                ptr = ptr + num_weights
                
                conv_weights = conv_weights.view_as(conv.weight.data)
                conv.weight.data.copy_(conv_weights)



In [9]:
model = Yolonet("cfg/yolov3.cfg")

In [10]:
model.load_weights("yolov3.weights")

In [11]:
inp_random = torch.rand(2,3,416,416)
pred_random = model(inp_random)
pred_random.shape

torch.Size([2, 10647, 85])

In [12]:
#model.header

In [13]:
#print(model.blocks)
#print(model.module_list)

In [14]:

def read_image(img_cv2, inp_dim):
    """
    
    converts and scale the image read by cv2 to 
    a tensor [batch,channel,inp_dim,inp_dim] for yolonet
    
    inputs: 
        -- img_cv2: shape is [h,w,channel]: the orignal image from cv2.imread
        -- inp_dim: integer, input size to yolo nueral network, e.g. 416
    Returns  
        -- img_net: tensor shape is [batch, channel, height, width]
        -- scale_factor
    """
    #img, scale_factor = (scale_image(img, (inp_dim, inp_dim)))
    
    img_cv2_h, img_cv2_w = img_cv2.shape[0], img_cv2.shape[1]
    w = inp_dim
    h = inp_dim
    scale_factor = min(w/img_cv2_w, h/img_cv2_h)
    new_w = int(img_cv2_w * scale_factor)
    new_h = int(img_cv2_h * scale_factor)
    resized_image = cv2.resize(img_cv2, (new_w,new_h), interpolation = cv2.INTER_CUBIC)
    # cv2.resize(img, (width, height), interpolation =...)
    
    canvas = np.full((inp_dim, inp_dim, 3), 128)

    
    canvas[0:new_h, 0:new_w, :] = resized_image  
    # align the resized image to the top-left of canvas
    
    img_net = canvas[:,:,::-1].transpose((2,0,1)).copy()   
    # canvas[:,:,::-1] reverse the number in color dimension BGR --> RGB
    # In CV2,channel order is [blue, green, red], but in PyTorch [red, green, blue]
    # transpose((2,0,1)): convert [H,W,C] to [C,H,W] for PyTorch
    
    img_net = torch.from_numpy(img_net).float().div(255.0).unsqueeze(0)   # add batch dimension 
    # img_net: [1, channel, height, width]
    return img_net, scale_factor

def load_classes(namesfile):
    fp = open(namesfile, "r")
    names = fp.read().split("\n")[:-1]
    return names



In [15]:
def bbox_iou(box1, box2):
    """
    inputs: 
        --box1: tensor shape [N1,4]: (x1,y1,x2,y2)
        --box2: tensor shape [N2,4]: (x1,y1,x2,y2)
            box1 and box2 should include the same number of boxes (N1=N2), 
            or one of them includes only one box and the other includes multiple boxes (N1=1, or N2=1)
        
    Returns  
        -- if N1 =N2, returns one-to-one IOUs
        -- if N1=1 or N2=1, returns one-to-many IOUs
    
    """
    #Get the coordinates of bounding boxes
    b1_x1, b1_y1, b1_x2, b1_y2 = box1[:,0], box1[:,1], box1[:,2], box1[:,3]
    b2_x1, b2_y1, b2_x2, b2_y2 = box2[:,0], box2[:,1], box2[:,2], box2[:,3]
    
    #get the corrdinates of the intersection rectangle
    inter_rect_x1 =  torch.max(b1_x1, b2_x1)
    inter_rect_y1 =  torch.max(b1_y1, b2_y1)
    inter_rect_x2 =  torch.min(b1_x2, b2_x2)
    inter_rect_y2 =  torch.min(b1_y2, b2_y2)
    
    #Intersection
    inter_area = torch.clamp(inter_rect_x2 - inter_rect_x1 + 1, min=0) * torch.clamp(inter_rect_y2 - inter_rect_y1 + 1, min=0)

    #Union
    b1_area = (b1_x2 - b1_x1 + 1)*(b1_y2 - b1_y1 + 1)
    b2_area = (b2_x2 - b2_x1 + 1)*(b2_y2 - b2_y1 + 1)
    
    #IOU
    iou = inter_area / (b1_area + b2_area - inter_area)
    
    return iou

In [16]:
#box1=torch.tensor([[, 0.1627, 0.1997, 0.7794]])
#box2=torch.tensor([[0.6983, 0.5865, 0.8460, 0.6061],[0.7983, 0.0461, 0.6084, 0.6684]])
#print("box1 is", box1)
#print("box2 is", box2)
#ious=bbox_iou(box2,box1)
#print("ious are", ious)

#tensor([[0.6983, 0.5865, 0.8460, 0.6061],
#        [0.7983, 0.0461, 0.6084, 0.6684]]) tensor([[0.7256, 0.1627, 0.1997, 0.7794]])

In [17]:

def non_max_suppression(prediction, confidence, num_classes, nms_conf = 0.4):
    """
    delivers all final bounding boxes, which are ready to draw on original images.
    inputs:
        -- prediction, from predict_transform() in Darkent model, includes all bounding boxes
           shape is [batch, (13x13+26x26+52x52)x3, 85]=[batch, 10647, 85], 10647 bounding boxes per batch
           prediction[0,0,:]: bx,by,bw,bh,sigmoid(t0),p1=sigmoid(), p2=sigmoid()..., p80=sigmoid()
        -- confidence, object confidence threshold (e.g. 0.5) for sigmoid(t0) in each bounding box
        -- num_classes, integer, the number of class (e.g. 80)
        -- nms_conf, the threshold of IOU for non-max suppression
    outputs:
        -- output, a tensor [N, 8], N is the total number of predicted boxes for the batch
                  each box: (batch_index, x1,y1,x2,y2,sigmoid(to), class probability, class index)
    """
    
    conf_mask = (prediction[:,:,4] > confidence).float().unsqueeze(2)
    #.unsqueeze(2) keep the dimension 2 to match the prediction dimension [batch, 10647,85]
    # conf_mask shape is [batch, 10647,1]
    
    prediction = prediction*conf_mask
    # clear all bounding boxes below confidence to zero-vectors
    # maintain all bounding boxes above confidence
    
    # convert (center,w,h) format to (top-left, bottom-right) format
    box_corner = prediction.new(prediction.shape)  #create a new tensor
    box_corner[:,:,0] = (prediction[:,:,0] - prediction[:,:,2]/2)
    box_corner[:,:,1] = (prediction[:,:,1] - prediction[:,:,3]/2)
    box_corner[:,:,2] = (prediction[:,:,0] + prediction[:,:,2]/2) 
    box_corner[:,:,3] = (prediction[:,:,1] + prediction[:,:,3]/2)
    prediction[:,:,:4] = box_corner[:,:,:4]
    # now, prediction in a corner format: [x1, y1, x1, y2, sigmoid(t0), sigmoid().....]
    
    
    batch_size = prediction.size(0)

    #initial first flag for output
    first = True
    

    for ind in range(batch_size):
        image_pred = prediction[ind]  # image_pred [10647,85]: prediction for one image
       #confidence threshholding 
       #NMS
    
        max_conf, max_conf_score = torch.max(image_pred[:,5:5+ num_classes], 1)
        max_conf = max_conf.float().unsqueeze(1)
        max_conf_score = max_conf_score.float().unsqueeze(1)
        # max_conf: max value, shape [10647,1]
        # max_conf_score: max index, shape [10647,1]
        
        seq = (image_pred[:,:5], max_conf, max_conf_score)
        image_pred = torch.cat(seq, 1) 
        #shape [10647,7], (x1,y1,x2,y2,sigmoid(to),pc_max,class_max)
        
        non_zero_ind =  (torch.nonzero(image_pred[:,4]))
        try:
            image_pred_ = image_pred[non_zero_ind.squeeze(),:].view(-1,7)
        except:
            continue
        
        if image_pred_.shape[0] == 0:
            continue     
        # image_pred_, includes only detected bounding boxes, shape [M1, 7]
    
        #Get the various classes detected in the image
        img_classes = torch.unique(image_pred_[:,-1])  
        # keep one element for each detected class.
        # img_class shape [M2], M2 < or = M1
        
        for cls in img_classes:
            #perform NMS for each class

            #get the detections with one particular class
            cls_mask = image_pred_*(image_pred_[:,-1] == cls).float().unsqueeze(1)
            class_mask_ind = torch.nonzero(cls_mask[:,-2]).squeeze()
            image_pred_class = image_pred_[class_mask_ind].view(-1,7) 
            # image_pred_class [M3,7]: detected bounding boxes for class cls
            
            #sort the detections such that the bounding box with the maximum objectness
            #confidence is at the beginning
            conf_sort_index = torch.sort(image_pred_class[:,4], descending = True )[1]
            image_pred_class = image_pred_class[conf_sort_index]
            idx = image_pred_class.size(0)   #Number of detected bounding boxes for class cls
            
            for i in range(idx):
                #Get the IOUs of all boxes that come after the one we are looking at 
                #in the loop
                try:
                    ious = bbox_iou(image_pred_class[i].unsqueeze(0), image_pred_class[i+1:])
                    # compute IOUs between box[i] and all the rest boxes
                except ValueError:
                    break
            
                except IndexError:
                    break
            
                #Zero out all the detections that have IoU > treshhold
                iou_mask = (ious < nms_conf).float().unsqueeze(1)
                image_pred_class[i+1:] *= iou_mask       
            
                #Remove the bounding boxes whose IOU with box[i] is greater than nms_conf
                non_zero_ind = torch.nonzero(image_pred_class[:,4]).squeeze()
                image_pred_class = image_pred_class[non_zero_ind].view(-1,7)
                
            batch_ind = image_pred_class.new(image_pred_class.size(0), 1).fill_(ind)      
            
            seq = (batch_ind, image_pred_class)  
            # attach the batch index to the bounding boxes for class cls
            
            if first:
                output = torch.cat(seq,1)
                first = False
            else:
                out = torch.cat(seq,1)
                output = torch.cat((output,out))
            # output shape [M4, 8], M4 is the number of valid bounding boxes 
            # up to current batch, image, class cls
            
        # output shape [M5, 8], M5 is the number of valid BBs up to current image
    
    try:
        return output
    except:
        return 0
    # the returned output [D,8], includes D bounding boxes for the batch
    # each box: (image_index, x1,y1,x2,y2,sigmoid(to), class probability, class index)

In [30]:
import os

if not os.path.exists("./results"):
    os.makedirs("./results")

inp_dim = 416
batch_size = 1
confidence = 0.5 # threshold for confidence score, default 0.5
nms_thresh = 0.4 # threshold for nms iou, default 0.4
num_classes = 80
classes = load_classes("data/coco.names")

model = Yolonet("cfg/yolov3.cfg")
model.load_weights("yolov3.weights")
model.eval()

#img = cv2.imread("dog-cycle-car.png")
img = cv2.imread("imgs/000017.jpg")
inp, scale_factor = read_image(img, inp_dim)
pred = model(inp)
prediction = non_max_suppression(pred, confidence, num_classes, nms_conf = nms_thresh)

prediction[:,1:5] /= scale_factor

for i in range(prediction.shape[0]):
    x = prediction[i]

    c1 = (x[1:3].int()).tolist() 
    c2 = (x[3:5].int()).tolist() 
    #img = results[int(x[0])]
    cls = int(x[-1])
    color = (5*cls,10*cls,100+cls)
    label = "{0}".format(classes[cls])
    cv2.rectangle(img, c1, c2,color, 2)
    t_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_PLAIN, 1 , 1)[0]
    c2 = c1[0] + t_size[0] + 50, c1[1] + t_size[1] + 4
    cv2.rectangle(img, c1, c2,color, -1)
    prob="(%.2f)" % round(float(x[5]*x[6]), 2)
    cv2.putText(img, label+str(prob), (c1[0], c1[1] + t_size[1] + 4), cv2.FONT_HERSHEY_PLAIN, 1, [2,2,2], 1);
   
cv2.imwrite("./results/000017.jpg", img)

True

In [31]:
print(prediction.shape)

torch.Size([2, 8])


In [28]:
prediction

tensor([[  0.0000, 180.4246,  63.6338, 285.5514, 176.0676,   0.9978,   0.9996,
           0.0000],
        [  0.0000,  93.0314,  78.7234, 382.8232, 319.7551,   0.9739,   0.9998,
          17.0000]])